# Phase 8 : Modélisation et Simulation Financière de Rentabilité Hôtelière (ROI)

Ce notebook présente la modélisation et la simulation de rentabilité (ROI) d'un investissement hôtelier au Maroc à l'approche de la Coupe du Monde FIFA 2030. Nous analysons l'impact du surcroît d'activité sur la rentabilité à travers les indicateurs de Valeur Actuelle Nette (NPV) et de Taux de Rentabilité Interne (IRR).

## 📖 Formules Mathématiques du Modèle

Le simulateur s'appuie sur les équations financières suivantes :

### 1. Indexation de l'ADR sur l'inflation
Le tarif journalier moyen (ADR) pour une année $t$ ($t = 0$ en 2026) s'exprime comme :
$$\text{ADR}_{\text{Base}, t} = \text{ADR}_0 \times (1 + i)^t$$
où $i$ désigne le taux d'inflation annuel moyen.

Pour l'année 2030 (Coupe du Monde), nous appliquons le boost $\beta$ :
$$\text{ADR}_{\text{WC}, t_{\text{2030}}} = \text{ADR}_{\text{Base}, t_{\text{2030}}} \times (1 + \beta)$$

### 2. Revenus Annuels d'Exploitation
$$\text{Revenu}_t = \text{Chambres} \times 365 \times \text{Taux d'occupation}_t \times \text{ADR}_t$$

### 3. Gross Operating Profit (GOP)
Le GOP est le cash flow annuel généré par l'hôtel après déduction des coûts opérationnels (marge de coût $\alpha$) :
$$\text{GOP}_t = \text{Revenu}_t \times (1 - \alpha)$$

### 4. Valeur Actuelle Nette (NPV)
La NPV mesure la richesse créée par le projet, actualisée au coût du capital $r$ (WACC) :
$$\text{NPV} = -I_0 + \sum_{t=1}^{N} \frac{\text{GOP}_t}{(1 + r)^t}$$
où $I_0$ représente l'investissement initial.

### 5. Taux de Rentabilité Interne (IRR)
L'IRR est le taux d'actualisation $r^*$ qui annule la NPV :
$$\text{NPV}(r^*) = -I_0 + \sum_{t=1}^{N} \frac{\text{GOP}_t}{(1 + r^*)^t} = 0$$

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ajouter le dossier parent au PYTHONPATH
sys.path.append(os.path.abspath('..'))

from src.roi_simulator import HotelROISimulator

## 1. Initialisation et Simulation des Scénarios

Nous simulons un hôtel 5 étoiles typique à Marrakech :
- **Taille** : 200 chambres
- **Investissement initial ($I_0$)** : 40 000 000 USD (construction et foncier)
- **ADR initial (2026)** : 250 USD
- **Taux d'occupation de base** : 68%
- **Taux d'actualisation (WACC)** : 8%
- **Marge GOP** : 35% (opex margin = 65%)
- **Inflation** : 2.5% par an
- **Choc Coupe du Monde 2030** : +40% ADR et occupation passant à 75% en 2030.

In [ ]:
# Instanciation du simulateur
simulator = HotelROISimulator(
    rooms=200,
    investment_usd=40000000.0,
    opex_margin=0.65,
    discount_rate=0.08,
    base_occupancy=0.68,
    wc_occupancy_2030=0.75,
    base_adr=250.0,
    wc_adr_boost_pct=0.40,
    inflation_rate=0.025
)

# Exécution de la simulation sur 10 ans (2026-2035)
df_roi = simulator.simulate_10years(start_year=2026)

# Affichage des colonnes clés pour inspecter les flux de trésorerie
cols_display = ['Year', 'ADR_Base_USD', 'Occ_Base', 'GOP_Base_USD', 'ADR_WC_USD', 'Occ_WC', 'GOP_WC_USD']
df_roi[cols_display].round(2)

## 2. Calcul et Analyse des Indicateurs Financiers

Calculons et comparons les performances financières globales des deux scénarios.

In [ ]:
metrics = simulator.calculate_metrics(df_roi)

print("==================================================")
print(" RÉSULTATS DE LA SIMULATION ROI HÔTELIÈRE")
print("==================================================")
print(f"Investissement Initial : 40.0 M$")
print(f"Coût du Capital (WACC)  : 8.0%\n")
print("--- SCÉNARIO A : BASE (Business as Usual) ---")
print(f"NPV (Valeur Actuelle Nette) : {metrics['NPV_Base_MUSD']:.2f} M$")
print(f"IRR (Taux de Rendement)     : {metrics['IRR_Base_Pct']:.2f}%")
print(f"Payback (Retour Invest.)     : {metrics['Payback_Base_Years']} ans")
print(f"ROI Cumulé sur 10 ans       : {metrics['ROI_Base_Pct']:.2f}%\n")
print("--- SCÉNARIO B : COUPE DU MONDE 2030 BOOST ---")
print(f"NPV (Valeur Actuelle Nette) : {metrics['NPV_WC_MUSD']:.2f} M$")
print(f"IRR (Taux de Rendement)     : {metrics['IRR_WC_Pct']:.2f}%")
print(f"Payback (Retour Invest.)     : {metrics['Payback_WC_Years']} ans")
print(f"ROI Cumulé sur 10 ans       : {metrics['ROI_WC_Pct']:.2f}%")
print("==================================================")

## 3. Visualisation de la Trajectoire des Cash Flows Cumulés

Le graphique suivant illustre le point d'équilibre (*breakeven*) et la divergence des deux scénarios.

In [ ]:
# Générer et afficher le tracé comparatif
%matplotlib inline
simulator.plot_comparison(df_roi, metrics)
plt.show()

## 4. Analyse de Sensibilité (Coût du Capital & ADR Boost)

Nous étudions la robustesse du projet sous différentes hypothèses de coût du capital (WACC) et d'ampleur du boost Coupe du Monde.

In [ ]:
rates = [0.06, 0.08, 0.10, 0.12]
boosts = [0.20, 0.40, 0.60]

print("Tableau de Sensibilité : NPV (Millions USD) du Scénario Coupe du Monde")
print("WACC \\ Boost |  20%   |  40%   |  60%   ")
print("------------------------------------------")
for r in rates:
    row_str = f" {r*100:4.1f}% WACC |"
    for b in boosts:
        test_sim = HotelROISimulator(discount_rate=r, wc_adr_boost_pct=b)
        df_test = test_sim.simulate_10years()
        test_metrics = test_sim.calculate_metrics(df_test)
        row_str += f"  {test_metrics['NPV_WC_MUSD']:5.2f} |"
    print(row_str)